# LSCI 220 Assignment 2
## Dominic Kelly, Id 198511370

# Introduction
A continuation of the processing done for Assignment 1.

# Beginning of material repeated from Assignment 1
Rip It Up was a New Zealand music magazine that was launched in 1977 and finally ceased publication in 2015. It was the closest thing that the New Zealand contemporary music scene had to a *New Musical Express* or *Rolling Stone*. In 1988, John Dix wrote in Stranded In Paradise (a history of New Zealand Rock 'n Roll) "New Zealand's best promotional outlet for what's happening out there in the real world of rock'n'roll is neither radio nor television, but *Rip It Up*, a monthly freebie that refuses to die."

On November 11, 2025, it was announced that the Papers Past website now holds more than 20 years of free, searchable and downloadable [content](https://paperspast.natlib.govt.nz/periodicals/rip-it-up) from Rip It Up for the years 1977-1998.

I downloaded all the album reviews for Rip It Up's first five months of publication (June - October 1977) and for the five months starting on its 20th anniversary.

I made sure that each review's metadata (artist, album title etc) was formatted consistently. Since Papers Past provides an image of the printed page as well as the generated text, I corrected any errors that I could tell had been created during the scanning process, but *not* grammatical or spelling errors that were apparent in the original.

In [1]:
# some preliminary imports, setting up static data etc
from ast import pattern
import nltk
import string
from nltk.corpus import PlaintextCorpusReader
from nltk import FreqDist
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import re

from enum import Enum

class Year(Enum):
    Y1977 = 0
    Y1997 = 1

punctuation_set = set(string.punctuation)

# additional punctuation are pieces of text that I found were appearing in the results 
# that I would like to have been removed with other punctuation
additional_punctuation = set(["'s", "'m", "'", '.', '"', "n't","'ve",
                              '’', '“', '”', '’;', '’’', '.)',
                              '–', '—', '...', '``', "''", '‘', 
                              '•', '”.', '",', '”,', '.):', '":', '’)',
                              '.”', ".'", '’,','’),', '–,', '—,', '’.', 
                              '(‘', '".', ').', '."', "')", '!!'])

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

nltk.download('wordnet')
nltk.download('omw-1.4') 

lemmatizer = WordNetLemmatizer()
corpus_root = '.'  # Directory containing text files
file_pattern = r'.*\.txt'  # Pattern to match .txt files
tokeniser = nltk.tokenize.WordPunctTokenizer()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\OEM\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\OEM\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\OEM\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


The ```do_scrubbing()``` function below does five types of pre-processing that I want to apply to the corpora. By default all are applied, which is how it is used below.

In [2]:
# define a few functions.

def do_scrubbing(tokens: list, remove_punctuation=True, remove_additional_puctuation=True, remove_stopwords=True, lowercase=True, lemmatize=True) -> list:
    # Most of the cleaning out of punctuation, stopwords, plural forms etc, all in one place.
    print("Initial token count:", len(tokens))
    if lowercase:
        tokens = [token.lower() for token in tokens]
    if (remove_punctuation):
        tokens = [word for word in tokens if word not in punctuation_set]
        # tokens = [token.translate(str.maketrans('', '', string.punctuation)) for token in tokens]

    if remove_additional_puctuation:
        tokens = [word for word in tokens if word not in additional_punctuation]

    if remove_stopwords:
        tokens = [word for word in tokens if word not in stop_words]

    if lemmatize:
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
    print("Final token count after scrubbing:", len(tokens))
    return tokens

# for analysing the text of the reviews, I prefer to exclude the headers that contain album and artist names, record labels and reviewer names
def remove_header_lines(body: str, pattern: str) -> str:
    lines = body.split('\n')
    reg_exp = re.compile(pattern)
    cleaned_lines = [line for line in lines if not reg_exp.search(line)]
    return '\n'.join(cleaned_lines)

def draw_wordcloud(frequencies: FreqDist, title: str):
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(dict(frequencies.most_common(20)))
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off") # Turn off the axis labels
    plt.title(title)
    plt.show()

def normalise_frequencies(frequencies: FreqDist, token_count: int) -> FreqDist:
    normalized_frequencies = FreqDist()
    for word, freq in frequencies.items():
        norm_freq = freq / token_count * 100
        normalized_frequencies[word] = norm_freq
    return normalized_frequencies

In [3]:
# now the actual work

corpus = PlaintextCorpusReader(corpus_root, file_pattern)

raw_review_corpus = [None, None]
review_corpus = [None, None]
review_tokens = [None, None]
scrubbed_tokens = [None, None]
frequencies = [None, None]
normalised_frequencies = [None, None]

# Access the text

raw_review_corpus[Year.Y1977.value] = corpus.raw(['RECORDS 77.06.01.txt', 'RECORDS 77.07.01.txt', 'RECORDS 77.08.01.txt', 'RECORDS 77.09.01.txt', 'RECORDS 77.10.01.txt'])
raw_review_corpus[Year.Y1997.value] = corpus.raw(['ALBUMS 97.06.01.txt', 'ALBUMS 97.07.01.txt', 'ALBUMS 97.08.01.txt', 'ALBUMS 97.09.01.txt', 'ALBUMS 97.10.01.txt'])

for year in [Year.Y1977.value, Year.Y1997.value]:
    review_corpus[year] = remove_header_lines(raw_review_corpus[year], r'^(Reviewed By:|Artist:|Album:|Record Label:).*$')
    review_tokens[year] = tokeniser.tokenize(review_corpus[year])
    scrubbed_tokens[year] = do_scrubbing(review_tokens[year])
    

Initial token count: 28021
Final token count after scrubbing: 12800
Initial token count: 38307
Final token count after scrubbing: 17775


Before looking at frequencies, a few simple metrics:

In [4]:
from nltk.text import Text
import pandas as pd
raw_review_tokens = [None, None]
headings = ['Year', 'Token Count', 'Review Count', 'Paragraph Count', 'Paragraphs per Review', 'Tokens per Review', 'TTR']
data = []
for year in [Year.Y1977.value, Year.Y1997.value]:
    token_type_count = len(set(scrubbed_tokens[year]))
    token_count = len(scrubbed_tokens[year])
    TTR = token_type_count / token_count
    paragraphs = review_corpus[year].split('\n')
    num_paragraphs = len(paragraphs)
    tokens_per_paragraph = token_count / num_paragraphs
    raw_review_tokens[year] = tokeniser.tokenize(raw_review_corpus[year])
    num_non_reviews = review_tokens[year].count('reviewed')
    num_reviews = raw_review_tokens[year].count('Reviewed') - num_non_reviews
    paragraphs_per_review = num_paragraphs / num_reviews
    tokens_per_review = token_count / num_reviews
    data.append([year, token_count, num_reviews, num_paragraphs, paragraphs_per_review, tokens_per_review, TTR])

df = pd.DataFrame(data, columns=headings )
print(df)

   Year  Token Count  Review Count  Paragraph Count  Paragraphs per Review  \
0     0        12800            57              419               7.350877   
1     1        17775           157              340               2.165605   

   Tokens per Review       TTR  
0         224.561404  0.324531  
1         113.216561  0.347229  


# End of Material Copied from Assignmet 1.

# Original Code Starts Here!

The most striking observation in my first look at Rip It Up's reviews from 1977 and 1997 was what seemed to be a move away from discussions on the technicalities of music and recording to mentions of genre.

There's a little bigram analysis that seems to reinforce that.

In [ ]:
for year in [Year.Y1977.value, Year.Y1997.value]:
    bigrams = list(nltk.bigrams(scrubbed_tokens[year]))
    bigrams_fdist = FreqDist(bigrams)
    print(f"Most common bigrams for year {Year(year).name}:")
    for freq in bigrams_fdist.most_common(10):
        print(freq)


Most common bigrams for year Y1977:
(('rock', 'n'), 19)
(('n', 'roll'), 18)
(('rhythm', 'section'), 12)
(('sound', 'like'), 11)
(('song', 'like'), 10)
(('first', 'album'), 10)
(('four', 'season'), 9)
(('side', 'one'), 8)
(('beach', 'boy'), 8)
(('dolly', 'parton'), 8)
Most common bigrams for year Y1997:
(('n', 'roll'), 20)
(('rock', 'n'), 19)
(('sound', 'like'), 16)
(('hip', 'hop'), 14)
(('punk', 'rock'), 12)
(('new', 'album'), 9)
(('yeah', 'yeah'), 8)
(('last', 'year'), 7)
(('liner', 'note'), 7)
(('pop', 'song'), 6)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\OEM\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


[('the', 'quick'), ('quick', 'brown'), ('brown', 'fox')]

While it's nice that both corpora have renditions of "rock 'n roll" at the top, "rhythm section" gives way to "hip hop" and "punk rock".


In [20]:
for year in [Year.Y1977.value, Year.Y1997.value]:
    trigrams = list(nltk.trigrams(scrubbed_tokens[year]))
    trigrams_fdist = FreqDist(trigrams)
    print(f"Most common trigrams for year {Year(year).name}:")
    for freq in trigrams_fdist.most_common(10):
        print(freq)

Most common trigrams for year Y1977:
(('rock', 'n', 'roll'), 18)
(('average', 'white', 'band'), 5)
(('day', 'dog', 'race'), 3)
(('time', 'love', 'hero'), 3)
(('miracle', 'billy', 'paul'), 3)
(('allman', 'brother', 'band'), 3)
(('rhythm', 'n', 'blue'), 3)
(('ben', 'e', 'king'), 3)
(('double', 'album', 'set'), 2)
(('good', 'rhythm', 'section'), 2)
Most common trigrams for year Y1997:
(('rock', 'n', 'roll'), 19)
(('drum', 'n', 'bass'), 5)
(('blue', 'sky', 'mar'), 3)
(('voodoo', 'glow', 'skull'), 3)
(('yeah', 'yeah', 'yeah'), 3)
(('plastic', 'ono', 'band'), 3)
(('rock', 'band', 'one'), 2)
(('solid', 'gold', 'hell'), 2)
(('album', 'tellin', 'story'), 2)
(('american', 'rock', 'n'), 2)


Trigrams are a little murkier with band names appearing frequently in both top tens. Fortunately "average white band" is a band name and not a descriptive phrase. 

*Corpora created using the [Rip It Up magazine archive at Papers Past](https://paperspast.natlib.govt.nz/periodicals/rip-it-up).*
    
*This archive is licensed for non-commercial use under a Creative Commons Attribution Non-Commercial Share Alike 3.0 (CC BY-NC-SA 3.0) licence. Rip it Up is not available for commercial use without the consent of Propeller Lamont Ltd.*